In [79]:
# ============================================================
# 1. Import libraries
# ============================================================

# Path is used to build safe file paths.
from pathlib import Path

# Pandas is used for data manipulation.
import pandas as pd

# NumPy is used for numerical transformations.
import numpy as np


print("Libraries imported successfully.")

Libraries imported successfully.


In [80]:
# ============================================================
# 2. Load the final cleaned and merged dataset
# ============================================================

# Build the path to the processed-data directory.
processed_directory = (
    Path.cwd().parent
    / "data"
    / "processed"
)

# Build the full path to the merged Parquet dataset.
data_path = (
    processed_directory
    / "cement_operations_merged.parquet"
)

# Load the final cleaned and integrated dataset.
df = pd.read_parquet(data_path)

# Confirm that the data loaded successfully.
print("Dataset shape:", df.shape)

df.head()

Dataset shape: (32880, 20)


,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity,opening_over_capacity_flag,closing_over_capacity_flag,available_inventory_tonnes,consumption_over_available_flag,unmet_demand_tonnes,stockout_flag,pour_fulfilment_rate,region,behavior
0,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,448,0,0,83.82,0,0.00,0,1.0000,North,aggressive
1,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,448,0,0,65.80,0,0.00,0,1.0000,North,aggressive
2,2022-01-07,SITE_001,CEM_I,33.60,33.60,0.00,47.72,14.12,0.90,4.48,448,0,0,47.72,0,0.00,0,1.0000,North,aggressive
3,2022-01-08,SITE_001,CEM_I,42.12,34.28,14.12,20.16,0.00,1.95,25.92,448,0,0,34.28,0,7.84,1,0.8139,North,aggressive
4,2022-01-09,SITE_001,CEM_I,0.00,0.00,0.00,34.38,34.38,1.42,16.81,448,0,0,34.38,0,0.00,0,NaN,North,aggressive


In [81]:
# ============================================================
# 3. Basic validation
# ============================================================

# Make sure the date column is stored as a datetime.
df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

# Check that the key columns exist.
required_columns = [
    "date",
    "site_id",
    "cement_type",
    "planned_pour_tonnes",
    "consumed_tonnes",
    "deliveries_tonnes",
    "opening_inventory_tonnes",
    "closing_inventory_tonnes",
    "rain_mm",
    "avg_temp_c",
    "silo_capacity",
    "region",
    "behavior"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("Required columns validated.")

Required columns validated.


In [82]:
# ============================================================
# 4. Sort the dataset
# ============================================================

# Sort observations so that records are chronological
# within each site and cement type.
#
# Correct ordering is essential before creating lag
# and rolling-window features.
df = df.sort_values(
    [
        "site_id",
        "cement_type",
        "date"
    ]
).reset_index(drop=True)

print("Dataset sorted successfully.")

Dataset sorted successfully.


In [83]:
# ============================================================
# 5. Define the forecasting groups
# ============================================================

# Each site and cement type represents a separate
# demand time series.
group_keys = [
    "site_id",
    "cement_type"
]

print("Forecasting groups defined:", group_keys)

Forecasting groups defined: ['site_id', 'cement_type']


In [84]:
# ============================================================
# 6. Identify potential leakage features
# ============================================================

# These variables depend directly on current-day consumption
# or on information that becomes known after consumption occurs.
#
# They are useful for EDA and reporting but should generally
# not be used as predictors when forecasting future
# consumed_tonnes.

leakage_columns = [
    "closing_inventory_tonnes",
    "consumption_over_available_flag",
    "unmet_demand_tonnes",
    "stockout_flag",
    "pour_fulfilment_rate"
]

print("Potential leakage features:")

for column in leakage_columns:
    print("-", column)

Potential leakage features:
- closing_inventory_tonnes
- consumption_over_available_flag
- unmet_demand_tonnes
- stockout_flag
- pour_fulfilment_rate


In [85]:
# ============================================================
# 7. Create calendar features
# ============================================================

# Extract the year.
df["year"] = df["date"].dt.year

# Extract the month number.
df["month"] = df["date"].dt.month

# Extract the calendar quarter.
df["quarter"] = df["date"].dt.quarter

# Extract the day of the month.
df["day_of_month"] = df["date"].dt.day

# Monday = 0 and Sunday = 6.
df["day_of_week"] = df["date"].dt.dayofweek

# Extract the ISO week number.
df["week_of_year"] = (
    df["date"]
    .dt
    .isocalendar()
    .week
    .astype("int16")
)

# Extract the day number within the year.
df["day_of_year"] = df["date"].dt.dayofyear

# Flag weekends.
df["is_weekend"] = (
    df["day_of_week"]
    .isin([5, 6])
    .astype("int8")
)

# Flag the first day of the month.
df["is_month_start"] = (
    df["date"]
    .dt
    .is_month_start
    .astype("int8")
)

# Flag the last day of the month.
df["is_month_end"] = (
    df["date"]
    .dt
    .is_month_end
    .astype("int8")
)

print("Calendar features created.")

Calendar features created.


In [86]:
# ============================================================
# 8. Create cyclical calendar features
# ============================================================

# Months and weekdays are cyclical.
# December is next to January, and Sunday is next to Monday.
#
# Sine and cosine encoding allows models to represent
# these circular relationships more naturally.

df["month_sin"] = np.sin(
    2 * np.pi * df["month"] / 12
)

df["month_cos"] = np.cos(
    2 * np.pi * df["month"] / 12
)

df["day_of_week_sin"] = np.sin(
    2 * np.pi * df["day_of_week"] / 7
)

df["day_of_week_cos"] = np.cos(
    2 * np.pi * df["day_of_week"] / 7
)

print("Cyclical calendar features created.")

Cyclical calendar features created.


In [87]:
# ============================================================
# 9. Create a complete daily calendar
# ============================================================

# Mark every row in the original dataset as an observed record.
# Any dates inserted during calendar expansion will later receive 0.
df["observed_record_flag"] = 1


# Create a function that expands each site/cement-type
# time series to a complete daily calendar.
def complete_daily_calendar(group):

    # Retrieve the site ID and cement type for this group.
    # group.name contains the values used by groupby().
    site_id, cement_type = group.name

    # Sort the existing observations chronologically.
    group = group.sort_values("date")

    # Create a continuous daily date range from the
    # first available date to the last available date.
    full_dates = pd.date_range(
        start=group["date"].min(),
        end=group["date"].max(),
        freq="D"
    )

    # Reindex the existing observations against the
    # complete daily calendar.
    #
    # Dates that did not previously exist will be inserted
    # and their operational variables will initially contain NaN.
    group = (
        group
        .set_index("date")
        .reindex(full_dates)
    )

    # Restore the date index name.
    group.index.name = "date"

    # Explicitly restore the site ID and cement type
    # for every row, including newly inserted dates.
    group["site_id"] = site_id
    group["cement_type"] = cement_type

    # Convert date back from the index into a normal column.
    group = group.reset_index()

    return group


# Apply the daily-calendar expansion separately to each
# site and cement-type time series.
df = (
    df
    .groupby(
        group_keys,
        group_keys=False
    )
    .apply(
        complete_daily_calendar,
        include_groups=False
    )
    .reset_index(drop=True)
)


# Newly inserted dates have no observed_record_flag.
# Convert these missing flags to 0.
#
# 1 = original observation
# 0 = inserted calendar date
df["observed_record_flag"] = (
    df["observed_record_flag"]
    .fillna(0)
    .astype("int8")
)


# Confirm that the calendar expansion completed successfully.
print("Daily calendar created.")
print("New shape:", df.shape)

# Confirm that the grouping identifiers still exist.
print("\nMissing site IDs:", df["site_id"].isna().sum())
print(
    "Missing cement types:",
    df["cement_type"].isna().sum()
)

# Show the number of original and inserted rows.
print("\nObserved vs inserted records:")
print(df["observed_record_flag"].value_counts())

Daily calendar created.
New shape: (98268, 35)

Missing site IDs: 0
Missing cement types: 0

Observed vs inserted records:
observed_record_flag
0    65388
1    32880
Name: count, dtype: int64


In [88]:
# ============================================================
# 10. Restore group identifiers
# ============================================================

# Because reindexing creates new rows, site_id and cement_type
# can become missing on inserted dates.
#
# Fill them within each continuous series.

df["site_id"] = (
    df["site_id"]
    .ffill()
    .bfill()
)

df["cement_type"] = (
    df["cement_type"]
    .ffill()
    .bfill()
)

In [89]:
# ============================================================
# 11. Identify original and inserted dates
# ============================================================

# Original rows already contain observed_record_flag = 1.
#
# Newly inserted calendar dates contain NaN, so replace those
# values with 0.
df["observed_record_flag"] = (
    df["observed_record_flag"]
    .fillna(0)
    .astype("int8")
)

print("Observed vs inserted records:")
print(
    df["observed_record_flag"]
    .value_counts()
)

Observed vs inserted records:
observed_record_flag
0    65388
1    32880
Name: count, dtype: int64


In [90]:
# ============================================================
# 12. Restore static site attributes
# ============================================================

# Region, behaviour and silo capacity are properties of a site.
# They can therefore be filled across inserted calendar dates
# without inventing demand values.

static_columns = [
    "region",
    "behavior",
    "silo_capacity"
]

for column in static_columns:

    df[column] = (
        df
        .groupby("site_id")[column]
        .transform(
            lambda series:
            series.ffill().bfill()
        )
    )


print("Missing static attributes:")
print(
    df[static_columns]
    .isna()
    .sum()
)

Missing static attributes:
region           0
behavior         0
silo_capacity    0
dtype: int64


In [91]:
# ============================================================
# 13. Recreate calendar features after daily expansion
# ============================================================

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter
df["day_of_month"] = df["date"].dt.day
df["day_of_week"] = df["date"].dt.dayofweek

df["week_of_year"] = (
    df["date"]
    .dt
    .isocalendar()
    .week
    .astype("int16")
)

df["day_of_year"] = df["date"].dt.dayofyear

df["is_weekend"] = (
    df["day_of_week"]
    .isin([5, 6])
    .astype("int8")
)

df["is_month_start"] = (
    df["date"]
    .dt
    .is_month_start
    .astype("int8")
)

df["is_month_end"] = (
    df["date"]
    .dt
    .is_month_end
    .astype("int8")
)

In [92]:
# ============================================================
# 14. Recreate cyclical calendar features
# ============================================================

df["month_sin"] = np.sin(
    2 * np.pi * df["month"] / 12
)

df["month_cos"] = np.cos(
    2 * np.pi * df["month"] / 12
)

df["day_of_week_sin"] = np.sin(
    2 * np.pi * df["day_of_week"] / 7
)

df["day_of_week_cos"] = np.cos(
    2 * np.pi * df["day_of_week"] / 7
)

print("Calendar features recreated.")

Calendar features recreated.


In [93]:
# ============================================================
# 15. Consumption lag features
# ============================================================

# Group actual consumption by site and cement type.
consumption_group = (
    df
    .groupby(
        group_keys,
        sort=False
    )["consumed_tonnes"]
)


# Define historical periods.
lag_periods = [
    1,
    7,
    14,
    28
]


# Create historical consumption features.
for lag in lag_periods:

    df[f"consumption_lag_{lag}"] = (
        consumption_group
        .shift(lag)
    )


print("Consumption lag features created.")

Consumption lag features created.


In [94]:
# ============================================================
# 16. Planned-pour lag features
# ============================================================

planned_group = (
    df
    .groupby(
        group_keys,
        sort=False
    )["planned_pour_tonnes"]
)

for lag in [1, 7, 14, 28]:

    df[f"planned_pour_lag_{lag}"] = (
        planned_group
        .shift(lag)
    )


print("Planned-pour lag features created.")

Planned-pour lag features created.


In [95]:
# ============================================================
# 17. Delivery lag features
# ============================================================

# Group deliveries by site and cement type.
delivery_group = (
    df
    .groupby(
        group_keys,
        sort=False
    )["deliveries_tonnes"]
)


# Previous-day and previous-week delivery quantities.
for lag in [1, 7]:

    df[f"deliveries_lag_{lag}"] = (
        delivery_group
        .shift(lag)
    )


print("Delivery lag features created.")

Delivery lag features created.


In [96]:
# ============================================================
# 18. Weather lag features
# ============================================================

# Historical rainfall.
rain_group = (
    df
    .groupby(
        group_keys,
        sort=False
    )["rain_mm"]
)


# Historical temperature.
temperature_group = (
    df
    .groupby(
        group_keys,
        sort=False
    )["avg_temp_c"]
)


df["rain_lag_1"] = (
    rain_group.shift(1)
)

df["rain_lag_7"] = (
    rain_group.shift(7)
)


df["temperature_lag_1"] = (
    temperature_group.shift(1)
)

df["temperature_lag_7"] = (
    temperature_group.shift(7)
)


print("Weather lag features created.")

Weather lag features created.


In [97]:
# ============================================================
# 19. Inventory lag features
# ============================================================

inventory_group = (
    df
    .groupby(
        group_keys,
        sort=False
    )["available_inventory_tonnes"]
)


df["available_inventory_lag_1"] = (
    inventory_group.shift(1)
)

df["available_inventory_lag_7"] = (
    inventory_group.shift(7)
)


print("Inventory lag features created.")

Inventory lag features created.


In [98]:
# ============================================================
# 20. Rolling consumption features - recommended version
# ============================================================

for window in [7, 14, 28]:

    df[f"consumption_rolling_mean_{window}"] = (
        df
        .groupby(
            group_keys,
            sort=False
        )["consumed_tonnes"]
        .transform(
            lambda series:
            series.shift(1)
            .rolling(
                window=window,
                min_periods=1
            )
            .mean()
        )
    )


print("Rolling consumption means created.")

Rolling consumption means created.


In [99]:
# ============================================================
# 21. Rolling consumption variability
# ============================================================

# Historical standard deviation can show whether recent
# demand has been stable or volatile.

for window in [7, 28]:

    df[f"consumption_rolling_std_{window}"] = (
        df
        .groupby(
            group_keys,
            sort=False
        )["consumed_tonnes"]
        .transform(
            lambda series:
            series.shift(1)
            .rolling(
                window=window,
                min_periods=2
            )
            .std()
        )
    )


print("Rolling variability features created.")

Rolling variability features created.


In [100]:
# ============================================================
# 22. Consumption change features
# ============================================================

# Compare yesterday's consumption with consumption
# one week earlier.
df["consumption_change_1_7"] = (
    df["consumption_lag_1"]
    - df["consumption_lag_7"]
)


# Compare consumption one week ago with consumption
# four weeks ago.
df["consumption_change_7_28"] = (
    df["consumption_lag_7"]
    - df["consumption_lag_28"]
)


print("Consumption change features created.")

Consumption change features created.


In [101]:
# ============================================================
# 23. Rolling delivery features
# ============================================================

# Calculate recent average deliveries using only
# information from previous days.
df["deliveries_rolling_mean_7"] = (
    df
    .groupby(
        group_keys,
        sort=False
    )["deliveries_tonnes"]
    .transform(
        lambda series:
        series.shift(1)
        .rolling(
            window=7,
            min_periods=1
        )
        .mean()
    )
)


print("Rolling delivery features created.")

Rolling delivery features created.


In [102]:
# ============================================================
# 23A. Verify engineered features
# ============================================================

# Check the overall shape of the feature-engineered dataset.
print("Dataset shape:", df.shape)

# Check whether important engineered columns exist.
expected_features = [
    "consumption_lag_1",
    "consumption_lag_7",
    "consumption_lag_14",
    "consumption_lag_28",
    "deliveries_lag_1",
    "deliveries_lag_7",
    "rain_lag_1",
    "rain_lag_7",
    "temperature_lag_1",
    "temperature_lag_7",
    "available_inventory_lag_1",
    "available_inventory_lag_7",
    "consumption_rolling_mean_7",
    "consumption_rolling_mean_14",
    "consumption_rolling_mean_28",
    "consumption_rolling_std_7",
    "consumption_rolling_std_28",
    "consumption_change_1_7",
    "consumption_change_7_28",
    "deliveries_rolling_mean_7"
]

missing_features = [
    column
    for column in expected_features
    if column not in df.columns
]

print("Missing engineered features:", missing_features)

Dataset shape: (98268, 59)
Missing engineered features: []


In [103]:
# ============================================================
# 23B. Verify calendar-based lag alignment
# ============================================================

# Select one site and one cement type so that we can
# manually inspect whether lag values align with the
# correct historical calendar dates.
lag_check = df.loc[
    (df["site_id"] == "SITE_001")
    & (df["cement_type"] == "CEM_I"),
    [
        "date",
        "observed_record_flag",
        "consumed_tonnes",
        "consumption_lag_1",
        "consumption_lag_7",
        "consumption_lag_14",
        "consumption_lag_28"
    ]
].head(40)

# Display the selected records.
lag_check

,date,observed_record_flag,consumed_tonnes,consumption_lag_1,consumption_lag_7,consumption_lag_14,consumption_lag_28
0,2022-01-02,1,45.26,NaN,NaN,NaN,NaN
1,2022-01-03,0,NaN,45.26,NaN,NaN,NaN
2,2022-01-04,1,33.16,NaN,NaN,NaN,NaN
3,2022-01-05,0,NaN,33.16,NaN,NaN,NaN
4,2022-01-06,0,NaN,NaN,NaN,NaN,NaN
5,2022-01-07,1,33.60,NaN,NaN,NaN,NaN
6,2022-01-08,1,34.28,33.60,NaN,NaN,NaN
7,2022-01-09,1,0.00,34.28,45.26,NaN,NaN
8,2022-01-10,0,NaN,0.00,NaN,NaN,NaN
9,2022-01-11,0,NaN,NaN,33.16,NaN,NaN


In [104]:
# ============================================================
# 24. Encode categorical variables
# ============================================================

# Convert selected categorical columns into one-hot
# encoded numerical variables.
categorical_columns = [
    "cement_type",
    "region",
    "behavior"
]

df = pd.get_dummies(
    df,
    columns=categorical_columns,
    prefix=categorical_columns,
    dtype="int8"
)


print("Categorical variables encoded.")

Categorical variables encoded.


In [105]:
# ============================================================
# 25. Final validation
# ============================================================

print("Final feature dataset shape:")
print(df.shape)


print("\nFirst five rows:")
display(df.head())


print("\nMissing values in lag/rolling features:")

feature_columns = [
    column
    for column in df.columns
    if (
        "lag_" in column
        or "rolling_" in column
    )
]

print(
    df[feature_columns]
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
    .head(20)
)

Final feature dataset shape:
(98268, 66)

First five rows:


,date,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity,opening_over_capacity_flag,...,cement_type_CEM_I,cement_type_CEM_II,cement_type_CEM_III,region_East,region_North,region_South,region_West,behavior_aggressive,behavior_chaotic,behavior_conservative
0,2022-01-02,45.26,45.26,63.85,19.97,38.56,3.23,14.28,448.0,0.0,...,1,0,0,0,1,0,0,1,0,0
1,2022-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,448.0,NaN,...,1,0,0,0,1,0,0,1,0,0
2,2022-01-04,33.16,33.16,47.06,18.74,32.64,8.25,14.23,448.0,0.0,...,1,0,0,0,1,0,0,1,0,0
3,2022-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,448.0,NaN,...,1,0,0,0,1,0,0,1,0,0
4,2022-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,448.0,NaN,...,1,0,0,0,1,0,0,1,0,0



Missing values in lag/rolling features:
consumption_lag_28             66283
planned_pour_lag_28            66283
planned_pour_lag_14            65867
consumption_lag_14             65867
planned_pour_lag_7             65654
consumption_lag_7              65654
temperature_lag_7              65654
available_inventory_lag_7      65654
deliveries_lag_7               65654
rain_lag_7                     65654
planned_pour_lag_1             65478
consumption_lag_1              65478
rain_lag_1                     65478
deliveries_lag_1               65478
temperature_lag_1              65478
available_inventory_lag_1      65478
consumption_rolling_std_7      25929
consumption_rolling_mean_7      5866
deliveries_rolling_mean_7       5866
consumption_rolling_mean_14      461
dtype: int64


In [106]:
# ============================================================
# 26. Verify true calendar lags
# ============================================================

# Inspect one specific site and cement type.
lag_check = df.loc[
    (
        df["site_id"] == "SITE_001"
    ),
    [
        "date",
        "site_id",
        "consumed_tonnes",
        "consumption_lag_1",
        "consumption_lag_7",
        "consumption_lag_14",
        "consumption_lag_28"
    ]
].head(40)


lag_check

,date,site_id,consumed_tonnes,consumption_lag_1,consumption_lag_7,consumption_lag_14,consumption_lag_28
0,2022-01-02,SITE_001,45.26,NaN,NaN,NaN,NaN
1,2022-01-03,SITE_001,NaN,45.26,NaN,NaN,NaN
2,2022-01-04,SITE_001,33.16,NaN,NaN,NaN,NaN
3,2022-01-05,SITE_001,NaN,33.16,NaN,NaN,NaN
4,2022-01-06,SITE_001,NaN,NaN,NaN,NaN,NaN
5,2022-01-07,SITE_001,33.60,NaN,NaN,NaN,NaN
6,2022-01-08,SITE_001,34.28,33.60,NaN,NaN,NaN
7,2022-01-09,SITE_001,0.00,34.28,45.26,NaN,NaN
8,2022-01-10,SITE_001,NaN,0.00,NaN,NaN,NaN
9,2022-01-11,SITE_001,NaN,NaN,33.16,NaN,NaN


In [108]:
# ============================================================
# 27. Create modelling dataset
# ============================================================

model_data = df.copy()


# Remove variables that would leak current-day outcome
# information into the prediction.
columns_to_exclude = [
    "closing_inventory_tonnes",
    "consumption_over_available_flag",
    "unmet_demand_tonnes",
    "stockout_flag",
    "pour_fulfilment_rate"
]


model_data = model_data.drop(
    columns=columns_to_exclude,
    errors="ignore"
)


print("Modelling dataset created.")
print("Shape:", model_data.shape)

Modelling dataset created.
Shape: (98268, 61)


In [109]:
# ============================================================
# 28. Keep observed records for supervised modelling
# ============================================================

# Train models only on dates where an actual operational
# observation existed.
#
# Rows with observed_record_flag = 0 were inserted only to
# create a continuous daily calendar for accurate lag features.
model_data = model_data[
    model_data["observed_record_flag"] == 1
].copy()


# Keep only records where the target variable is known.
# consumed_tonnes is the target that the forecasting model
# will eventually predict.
model_data = model_data[
    model_data["consumed_tonnes"].notna()
].copy()


# Reset the index after filtering so that the modelling
# dataset has a clean sequential index.
model_data = model_data.reset_index(drop=True)


# Confirm the number of genuine observations available
# for supervised modelling.
print(
    "Observed modelling rows:",
    len(model_data)
)

# Display the final modelling dataset dimensions.
print(
    "Modelling dataset shape:",
    model_data.shape
)

Observed modelling rows: 32880
Modelling dataset shape: (32880, 61)


In [110]:
# ============================================================
# 29. Check missing lag and rolling features in modelling data
# ============================================================

# Identify all engineered lag and rolling-window features.
lag_rolling_columns = [
    column
    for column in model_data.columns
    if (
        "lag_" in column
        or "rolling_" in column
    )
]


# Calculate the number of missing values for each feature.
missing_count = (
    model_data[lag_rolling_columns]
    .isna()
    .sum()
)


# Calculate the percentage of missing values for each feature.
missing_percentage = (
    model_data[lag_rolling_columns]
    .isna()
    .mean()
    .mul(100)
    .round(2)
)


# Combine the counts and percentages into one summary table.
missing_feature_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_pct": missing_percentage
})


# Sort the results so that features with the highest
# percentage of missing values appear first.
missing_feature_summary = (
    missing_feature_summary
    .sort_values(
        "missing_pct",
        ascending=False
    )
)


# Display the missing-value summary.
display(missing_feature_summary)

,missing_count,missing_pct
consumption_lag_28,22229,67.61
planned_pour_lag_28,22229,67.61
planned_pour_lag_14,22047,67.05
consumption_lag_14,22047,67.05
planned_pour_lag_1,21998,66.90
consumption_lag_1,21998,66.90
temperature_lag_1,21998,66.90
available_inventory_lag_1,21998,66.90
deliveries_lag_1,21998,66.90
rain_lag_1,21998,66.90


In [112]:
# ============================================================
# 30. Examine observation frequency by site and cement type
# ============================================================

# Create a temporary copy for the frequency analysis.
frequency_check = df[
    df["observed_record_flag"] == 1
].copy()


# Reconstruct the original cement_type category
# from the one-hot encoded cement-type columns.
frequency_check["cement_type"] = np.select(
    [
        frequency_check["cement_type_CEM_I"] == 1,
        frequency_check["cement_type_CEM_II"] == 1,
        frequency_check["cement_type_CEM_III"] == 1
    ],
    [
        "CEM_I",
        "CEM_II",
        "CEM_III"
    ],
    default="UNKNOWN"
)


# Sort genuine observations by site, cement type and date.
frequency_check = (
    frequency_check
    .sort_values(
        ["site_id", "cement_type", "date"]
    )
    .copy()
)


# Calculate the number of calendar days since the previous
# observation for the same site and cement type.
frequency_check["days_since_previous_observation"] = (
    frequency_check
    .groupby(
        ["site_id", "cement_type"]
    )["date"]
    .diff()
    .dt.days
)


# Summarise the gaps between consecutive observations.
observation_gap_summary = (
    frequency_check[
        "days_since_previous_observation"
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95
        ]
    )
)


print("Observation-gap summary:")
print(observation_gap_summary)


# Display the most common calendar gaps.
print("\nMost common gaps between observations:")

print(
    frequency_check[
        "days_since_previous_observation"
    ]
    .value_counts()
    .sort_index()
    .head(15)
)

Observation-gap summary:
count    32790.000000
mean         2.994145
std          2.449097
min          1.000000
25%          1.000000
50%          2.000000
75%          4.000000
90%          6.000000
95%          8.000000
max         27.000000
Name: days_since_previous_observation, dtype: float64

Most common gaps between observations:
days_since_previous_observation
1.0     10882
2.0      7395
3.0      4855
4.0      3279
5.0      2078
6.0      1448
7.0       940
8.0       633
9.0       410
10.0      304
11.0      186
12.0      141
13.0       82
14.0       50
15.0       25
Name: count, dtype: int64


In [113]:
# ============================================================
# 31. Create previous-observation lag features
# ============================================================

# Work only with genuine operational observations.
# These lags refer to the previous recorded observation
# for the same site and cement type rather than an exact
# calendar-day offset.

observed_data = (
    df[df["observed_record_flag"] == 1]
    .copy()
)

# Reconstruct cement_type if one-hot encoding has already been applied.
observed_data["cement_type"] = np.select(
    [
        observed_data["cement_type_CEM_I"] == 1,
        observed_data["cement_type_CEM_II"] == 1,
        observed_data["cement_type_CEM_III"] == 1
    ],
    [
        "CEM_I",
        "CEM_II",
        "CEM_III"
    ],
    default="UNKNOWN"
)

# Sort records so previous observations are chronologically correct.
observed_data = observed_data.sort_values(
    ["site_id", "cement_type", "date"]
).reset_index(drop=True)

# Define the grouping columns.
observation_group_keys = [
    "site_id",
    "cement_type"
]

# Create previous-observation consumption features.
# prev_1 = most recent previous recorded consumption.
# prev_2 = second previous recorded consumption.
# prev_3 = third previous recorded consumption.
for lag in [1, 2, 3, 5]:
    observed_data[f"consumption_prev_{lag}"] = (
        observed_data
        .groupby(observation_group_keys)["consumed_tonnes"]
        .shift(lag)
    )

print("Previous-observation lag features created.")

Previous-observation lag features created.


In [114]:
# Check how complete the previous-observation lag features are.
previous_lag_columns = [
    "consumption_prev_1",
    "consumption_prev_2",
    "consumption_prev_3",
    "consumption_prev_5"
]

previous_lag_missing = pd.DataFrame({
    "missing_count": observed_data[previous_lag_columns].isna().sum(),
    "missing_pct": (
        observed_data[previous_lag_columns]
        .isna()
        .mean()
        .mul(100)
        .round(2)
    )
})

previous_lag_missing

,missing_count,missing_pct
consumption_prev_1,90,0.27
consumption_prev_2,180,0.55
consumption_prev_3,270,0.82
consumption_prev_5,450,1.37


In [115]:
# ============================================================
# 32. Build the final modelling dataset
# ============================================================

# Start from observed_data because it contains:
# - only genuine operational observations;
# - the new previous-observation lag features;
# - the rolling features created earlier;
# - the encoded categorical variables already present in df.

final_model_data = observed_data.copy()


# Define the sparse exact-calendar lag features.
# These had approximately 66%–68% missing values,
# so they are not ideal for the final model.
sparse_exact_lag_columns = [
    "consumption_lag_1",
    "consumption_lag_7",
    "consumption_lag_14",
    "consumption_lag_28",
    "planned_pour_lag_1",
    "planned_pour_lag_7",
    "planned_pour_lag_14",
    "planned_pour_lag_28",
    "deliveries_lag_1",
    "deliveries_lag_7",
    "rain_lag_1",
    "rain_lag_7",
    "temperature_lag_1",
    "temperature_lag_7",
    "available_inventory_lag_1",
    "available_inventory_lag_7"
]


# Define potential leakage variables.
# These contain current-day outcome information and should
# not be used as predictors of future consumed_tonnes.
leakage_columns = [
    "closing_inventory_tonnes",
    "consumption_over_available_flag",
    "unmet_demand_tonnes",
    "stockout_flag",
    "pour_fulfilment_rate"
]


# Combine the columns that should be removed.
columns_to_remove = (
    sparse_exact_lag_columns
    + leakage_columns
)


# Remove the sparse and leakage-prone features.
# errors="ignore" prevents failure if any column
# has already been removed.
final_model_data = final_model_data.drop(
    columns=columns_to_remove,
    errors="ignore"
)


# Reset the index after preparing the final modelling table.
final_model_data = (
    final_model_data
    .reset_index(drop=True)
)


# Display the resulting dimensions.
print(
    "Final modelling dataset shape:",
    final_model_data.shape
)

Final modelling dataset shape: (32880, 50)


In [116]:
# ============================================================
# 33. Verify final modelling features
# ============================================================

# Confirm that the preferred previous-observation features exist.
preferred_features = [
    "consumption_prev_1",
    "consumption_prev_2",
    "consumption_prev_3",
    "consumption_prev_5",
    "consumption_rolling_mean_7",
    "consumption_rolling_mean_14",
    "consumption_rolling_mean_28",
    "consumption_rolling_std_7",
    "consumption_rolling_std_28",
    "deliveries_rolling_mean_7"
]


# Identify any expected features that are missing.
missing_preferred_features = [
    column
    for column in preferred_features
    if column not in final_model_data.columns
]

print(
    "Missing preferred features:",
    missing_preferred_features
)

Missing preferred features: []


In [117]:
# ============================================================
# 34. Check remaining missing values
# ============================================================

# Calculate missing-value counts and percentages
# for every column in the final modelling dataset.
final_missing_summary = pd.DataFrame({
    "missing_count": (
        final_model_data
        .isna()
        .sum()
    ),
    "missing_pct": (
        final_model_data
        .isna()
        .mean()
        .mul(100)
        .round(2)
    )
})


# Keep only columns that still contain missing values.
final_missing_summary = (
    final_missing_summary[
        final_missing_summary["missing_count"] > 0
    ]
    .sort_values(
        "missing_pct",
        ascending=False
    )
)


display(final_missing_summary)

,missing_count,missing_pct
consumption_change_7_28,29273,89.03
consumption_change_1_7,29204,88.82
consumption_rolling_std_7,8675,26.38
consumption_rolling_mean_7,2003,6.09
deliveries_rolling_mean_7,2003,6.09
consumption_prev_5,450,1.37
consumption_prev_3,270,0.82
consumption_rolling_mean_14,197,0.60
consumption_rolling_std_28,186,0.57
consumption_prev_2,180,0.55


In [118]:
# ============================================================
# 35. Remove highly sparse change features
# ============================================================

# These change features were derived from the exact-calendar
# lag variables, which were highly sparse because individual
# site/cement combinations are not observed every day.
#
# With almost 90% missing values, these features are unlikely
# to provide reliable information to the forecasting model.

high_missing_features = [
    "consumption_change_1_7",
    "consumption_change_7_28"
]

final_model_data = final_model_data.drop(
    columns=high_missing_features,
    errors="ignore"
)

print("Highly sparse change features removed.")

print(
    "Final modelling dataset shape:",
    final_model_data.shape
)

Highly sparse change features removed.
Final modelling dataset shape: (32880, 48)


In [119]:
# ============================================================
# 36. Examine availability of rolling features
# ============================================================

rolling_features = [
    "consumption_rolling_mean_7",
    "consumption_rolling_mean_14",
    "consumption_rolling_mean_28",
    "consumption_rolling_std_7",
    "consumption_rolling_std_28",
    "deliveries_rolling_mean_7"
]

rolling_availability = pd.DataFrame({
    "available_count": (
        final_model_data[rolling_features]
        .notna()
        .sum()
    ),
    "available_pct": (
        final_model_data[rolling_features]
        .notna()
        .mean()
        .mul(100)
        .round(2)
    )
})

rolling_availability

,available_count,available_pct
consumption_rolling_mean_7,30877,93.91
consumption_rolling_mean_14,32683,99.40
consumption_rolling_mean_28,32790,99.73
consumption_rolling_std_7,24205,73.62
consumption_rolling_std_28,32694,99.43
deliveries_rolling_mean_7,30877,93.91


In [120]:
# ============================================================
# 37. Remove sparse 7-day rolling variability feature
# ============================================================

# The 7-day rolling standard deviation is missing for
# approximately 26% of genuine modelling observations.
#
# This occurs because many site/cement-type series do not
# contain enough observations within a seven-day window
# to calculate a reliable standard deviation.
#
# The 28-day rolling standard deviation is retained because
# it provides a similar measure of demand variability while
# being available for more than 99% of observations.

final_model_data = final_model_data.drop(
    columns=["consumption_rolling_std_7"],
    errors="ignore"
)

print(
    "7-day rolling standard deviation removed."
)

print(
    "Final modelling dataset shape:",
    final_model_data.shape
)

7-day rolling standard deviation removed.
Final modelling dataset shape: (32880, 47)


In [121]:
# ============================================================
# 38. Final missing-value check
# ============================================================

# Calculate remaining missing-value counts and percentages.
remaining_missing = pd.DataFrame({
    "missing_count": final_model_data.isna().sum(),
    "missing_pct": (
        final_model_data
        .isna()
        .mean()
        .mul(100)
        .round(2)
    )
})


# Keep only columns that still contain missing values.
remaining_missing = (
    remaining_missing[
        remaining_missing["missing_count"] > 0
    ]
    .sort_values(
        "missing_pct",
        ascending=False
    )
)

display(remaining_missing)

,missing_count,missing_pct
consumption_rolling_mean_7,2003,6.09
deliveries_rolling_mean_7,2003,6.09
consumption_prev_5,450,1.37
consumption_prev_3,270,0.82
consumption_rolling_mean_14,197,0.60
consumption_rolling_std_28,186,0.57
consumption_prev_2,180,0.55
consumption_rolling_mean_28,90,0.27
consumption_prev_1,90,0.27


In [122]:
# ============================================================
# 39. Create complete training records
# ============================================================

# Define the historical features that will be retained
# for the forecasting model.
#
# These features have relatively low missingness and provide
# information about recent consumption and delivery behaviour.
required_history_features = [
    "consumption_prev_1",
    "consumption_prev_2",
    "consumption_prev_3",
    "consumption_prev_5",
    "consumption_rolling_mean_7",
    "consumption_rolling_mean_14",
    "consumption_rolling_mean_28",
    "consumption_rolling_std_28",
    "deliveries_rolling_mean_7"
]


# Record the number of observations before removing rows
# without sufficient historical information.
rows_before = len(final_model_data)


# Keep only observations where all selected historical
# features are available.
final_model_data = (
    final_model_data
    .dropna(
        subset=required_history_features
    )
    .reset_index(drop=True)
)


# Record the final number of modelling observations.
rows_after = len(final_model_data)


# Calculate how many observations were removed.
rows_removed = rows_before - rows_after


# Calculate the percentage of observations removed.
rows_removed_pct = (
    rows_removed
    / rows_before
    * 100
)


print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Rows removed:", rows_removed)

print(
    "Percentage removed:",
    f"{rows_removed_pct:.2f}%"
)

Rows before: 32880
Rows after: 30542
Rows removed: 2338
Percentage removed: 7.11%


In [123]:
# ============================================================
# 40. Verify final modelling dataset
# ============================================================

# Count all remaining missing values.
total_missing = (
    final_model_data
    .isna()
    .sum()
    .sum()
)

print(
    "Total remaining missing values:",
    total_missing
)

print(
    "Final modelling dataset shape:",
    final_model_data.shape
)

Total remaining missing values: 0
Final modelling dataset shape: (30542, 47)


In [124]:
# ============================================================
# 41. Save feature-engineered datasets
# ============================================================

# Define output paths for the complete feature dataset
# and the final model-ready dataset.
feature_output_path = (
    processed_directory
    / "cement_operations_features.parquet"
)

model_output_path = (
    processed_directory
    / "cement_operations_model_ready.parquet"
)


# Save the complete feature-engineered dataset.
#
# This preserves the expanded calendar and all engineered
# features for future analysis or alternative modelling.
df.to_parquet(
    feature_output_path,
    index=False
)


# Save the final modelling dataset.
#
# This dataset contains genuine operational observations,
# selected historical features, no target-leakage variables,
# and no missing values in the retained features.
final_model_data.to_parquet(
    model_output_path,
    index=False
)


print("Feature-engineered datasets saved successfully.")

print("\nFull feature dataset:")
print(feature_output_path)

print("\nModel-ready dataset:")
print(model_output_path)

print(
    "\nModel-ready shape:",
    final_model_data.shape
)

Feature-engineered datasets saved successfully.

Full feature dataset:
c:\Users\User.DESKTOP-775\Documents\mig-cement-demand-forecasting-E\data\processed\cement_operations_features.parquet

Model-ready dataset:
c:\Users\User.DESKTOP-775\Documents\mig-cement-demand-forecasting-E\data\processed\cement_operations_model_ready.parquet

Model-ready shape: (30542, 47)


In [ ]:
# ============================================================
# 42. Verify saved files
# ============================================================

print(
    "Feature dataset exists:",
    feature_output_path.exists()
)

print(
    "Model-ready dataset exists:",
    model_output_path.exists()
)

Feature dataset exists: True
Model-ready dataset exists: True
